# 06 · Function Tools: Uncle Uses YOUR Python 🧰

A **function tool** is a normal Python function that the AI can ask us to run.

```
You:   "How much is kopi-o?"
AI:    "Please run get_menu() for me"        <- a function_call
We:    run get_menu(), send back the menu     <- a function_call_output
AI:    "Kopi-O is $1.20 lah!"
```
The AI never runs your code itself — **you** run it and send back the answer.

In [ ]:
import os
from getpass import getpass

from dotenv import load_dotenv
from openai import OpenAI

if not os.path.exists("main.py"):   # make sure we're in the project folder
    os.chdir("../..")

load_dotenv()
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Paste your OpenAI API key: ")

client = OpenAI()
MODEL = "gpt-6-luna"
print("Ready!")

## 1. The menu (with pandas)

In [ ]:
import pandas as pd

menu = pd.read_csv("menu.csv")
menu

In [ ]:
# filter rows: only drinks
menu[menu["category"] == "drink"]

In [ ]:
def get_menu():
    return menu.to_string(index=False)


print(get_menu())

## 2. Describe the tool for the AI
The AI can't see your code. It only sees a **name** and a **description** — so write them clearly!

In [ ]:
tools = [
    {
        "type": "function",
        "name": "get_menu",
        "description": "Get the kopitiam menu with prices.",
        "parameters": {"type": "object", "properties": {}},   # no inputs needed
    },
]

uncle = "You are a friendly kopitiam uncle. Use get_menu for prices and never guess. Keep replies short."

## 3. The AI asks for the tool

In [ ]:
response = client.responses.create(model=MODEL, instructions=uncle, tools=tools, input="How much is a kopi-o?")

for item in response.output:
    print(item.type)

In [ ]:
call = response.output[-1]   # the function_call is the last item

print("Name:", call.name)
print("Inputs:", call.arguments)
print("Call ID:", call.call_id)

## 4. We run it and send the answer back
`previous_response_id` tells the AI *"continue from your last response"*.

In [ ]:
result = get_menu()

response = client.responses.create(
    model=MODEL,
    instructions=uncle,
    tools=tools,
    previous_response_id=response.id,
    input=[{"type": "function_call_output", "call_id": call.call_id, "output": result}],
)

print(response.output_text)

## 5. A tool with an input: the weather 🌦️
data.gov.sg gives a free 2-hour forecast for every area in Singapore — no key needed.

In [ ]:
import requests


def get_weather(area):
    data = requests.get("https://api-open.data.gov.sg/v2/real-time/api/two-hr-forecast").json()
    for forecast in data["data"]["items"][0]["forecasts"]:
        if area.lower() in forecast["area"].lower():
            return f"{forecast['area']}: {forecast['forecast']}"
    return f"No area called {area}"


print(get_weather("Bedok"))

In [ ]:
tools.append({
    "type": "function",
    "name": "get_weather",
    "description": "Get the live weather forecast for a Singapore area, e.g. 'Bedok' or 'City'.",
    "parameters": {
        "type": "object",
        "properties": {"area": {"type": "string"}},   # the input the AI must give us
        "required": ["area"],
    },
})

## 6. Put it together: a loop
The AI might ask for several tools in a row, so we keep going **while** it asks for functions.
This is the same code as part 4 of `main.py`.

In [ ]:
import json

functions = {"get_menu": get_menu, "get_weather": get_weather}   # name the AI uses -> our function


def ask(text):
    response = client.responses.create(model=MODEL, instructions=uncle, tools=tools, input=text)

    while any(item.type == "function_call" for item in response.output):
        results = []
        for item in response.output:
            if item.type == "function_call":
                arguments = json.loads(item.arguments)          # text -> dictionary (notebook 02!)
                output = functions[item.name](**arguments)       # run OUR function
                results.append({"type": "function_call_output", "call_id": item.call_id, "output": output})
                print("🔧", item.name, arguments)

        response = client.responses.create(
            model=MODEL, instructions=uncle, tools=tools, previous_response_id=response.id, input=results
        )

    return response.output_text


print(ask("I'm in Bedok. Hot or not? What should I drink, and how much?"))

## ✏️ Exercises
1. Add **Teh Halia** (ginger tea, $1.60) to `menu.csv`, re-run the cells, and ask Uncle about it.
2. Write a `get_time()` tool so Uncle says *"Good morning"* or *"Good evening"* correctly (hint: `datetime.now()`).
3. 🌶️ Challenge: a `recommend_drink(weather)` tool with your own rules.

In [ ]:
# your code here